In [ ]:
#Import of Libraries
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import sentencepiece
import google.protobuf
import files

/home/abdallah/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
#Reading the datasets and taking the two columns we need and renaming them to text/target

#Dataset_1-Training
SumArabic_tr = pd.read_json("sumarabic-1.0-train.jsonl" , lines = True)
SumArabic_tr = SumArabic_tr[["text" , "headline"]].rename(columns = {"headline" : "target"})

#Dataset_1-Testing
SumArabic_te = pd.read_json("sumarabic-1.0-test.jsonl" , lines = True)
SumArabic_te = SumArabic_te[["text" , "headline"]].rename(columns = {"headline" : "target"})

#Dataset_1-Validation
SumArabic_va = pd.read_json("sumarabic-1.0-valid.jsonl" , lines = True)
SumArabic_va = SumArabic_va[["text" , "headline"]].rename(columns = {"headline" : "target"})

#Dataset_2
AraSum = pd.read_csv("AraSum.csv" , sep = "\t" , engine = "python" , header = None , names = ["index" , "text" , "target"]) 
AraSum = AraSum.drop("index" , axis = 1)

#Dataset_3
EgyD = pd.read_parquet("train-00000-of-00001.parquet" , engine = "pyarrow")
EgyD = EgyD[["text" , "summarized_text"]].rename(columns = {"summarized_text" : "target"})

#Dataset_4
ArabicKagg = pd.read_csv("summarizdataset.csv")
ArabicKagg = ArabicKagg[["text" , "summarizer"]].rename(columns = {"summarizer" : "target"})




In [3]:
#Splitting the datasets into train/test

#First we concat the three unsplit datasets
df = pd.concat([AraSum , EgyD , ArabicKagg] , ignore_index=True)
df_train , df_test = train_test_split(df , test_size=0.15 , random_state=42)
df_train , df_valid = train_test_split(df_train , test_size = 0.15 , random_state = 42)

#We then add the already split dataset into the split
df_train = pd.concat([df_train , SumArabic_tr] , ignore_index=True)
df_test = pd.concat([df_test , SumArabic_te] , ignore_index=True)
df_valid = pd.concat([df_valid, SumArabic_va] , ignore_index=True)



In [4]:
#Function for Cleaning
def cleaning_words(x):
    x = re.sub(r' +', ' ', x)
    x = re.sub(r'"([^"]+)"', r'\1', x, flags=re.UNICODE)
    x = re.sub(r'[»«]', '', x)
    x = re.sub(r'[a-zA-Z]', '', x)
    x = re.sub(r'\(\d+\s*/\s*\w+\)', '', x) 
    x = re.sub(r'[()/]', '', x)
    x = re.sub(r'[ء-ي]\.[ء-ي]', '', x)
    x = re.sub(r'\s[ء-ي]\s', ' ', x)
    x = re.sub(r'\n', '', x)
    x = x.replace('\xa0', ' ')
    x = x.replace('\u200f', '')
    x = x.replace('\u200e', '')
    x = x.replace('\ufeff', '')
    x = x.replace('\u200c', '')
    x = x.replace('\u200b', '')
    x = x.replace('\u202c', '')
    x = x.replace('\u202a', '')
    x = x.replace('\u202b', '')
    x = x.replace('…', '.')
    x = x.replace('٪', '%')
    x = x.replace('٬', '')
    x = x.replace('―', '-')
    return x

In [ ]:
#Calling the function to clean text and target
df_train["processed_data"] = df_train["text"].map(cleaning_words)
df_test["processed_data"] = df_test["text"].map(cleaning_words)
df_valid["processed_data"] = df_valid["text"].map(cleaning_words)

df_train["processed_target"] = df_train["target"].map(cleaning_words)
df_test["processed_target"] = df_test["target"].map(cleaning_words)
df_valid["processed_target"] = df_valid["target"].map(cleaning_words)



In [ ]:
#Function for Tokenization for both text and target
tokenizer = AutoTokenizer.from_pretrained('moussaKam/AraBART')

def tokenize(batch):
    inputs = tokenizer(
        batch['processed_data'],
        max_length=1024,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        batch['processed_target'],
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

In [ ]:
#Calling the function to tokenize the data (same function does text and target)
df_train = df_train.apply(tokenize , axis = 1)
df_test = df_test.apply(tokenize , axis = 1)
df_valid = df_valid.apply(tokenize , axis = 1)

In [ ]:
#Making it into a df after the apply axis = 1
df_train = pd.DataFrame(df_train.tolist())
df_valid = pd.DataFrame(df_valid.tolist())
df_test = pd.DataFrame(df_test.tolist())



In [9]:
df_train

,attention_mask,input_ids,labels
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[11136, 1684, 2365, 1875, 23230, 846, 538, 19,...","[1684, 11136, 2365, 1875, 23230, 19, 2039, 240..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[74, 397, 391, 4, 2482, 5588, 28, 7185, 14880,...","[297, 2991, 410, 5588, 28, 7185, 4, 8964, 2593..."
2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[809, 586, 1728, 6060, 76, 541, 4, 21041, 20, ...","[809, 586, 1728, 6060, 76, 541, 4, 21041, 20, ..."
3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[25, 3780, 382, 157, 453, 15412, 5267, 15685, ...","[27, 59, 31215, 5, 768, 453, 191, 2865, 7588, ..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1127, 119, 3519, 1530, 19, 2441, 4184, 19338,...","[12929, 3133, 2650, 19, 3519, 192, 8156, 49, 3..."
...,...,...,...
46906,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[8439, 891, 3742, 31457, 76, 148, 11113, 11412...","[1411, 14409, 4401, 2869, 8, 8405, 25014, 6, 5..."
46907,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[5261, 1854, 9149, 40, 6266, 400, 42125, 45, 6...","[3073, 677, 110, 292, 6451, 833, 14405, 659, 1..."
46908,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[811, 740, 3400, 8395, 76, 40, 2451, 11324, 19...","[767, 15243, 4812, 3602, 17, 1212, 6946, 8, 13..."
46909,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[929, 135, 9615, 13, 5453, 20, 477, 3734, 1981...","[9615, 13, 5453, 2354, 3734, 8503, 253, 5, 131..."
